<a href="https://colab.research.google.com/github/Rakesh-562/ScoreStats/blob/main/Random_forest_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install scikit-learn

In [28]:

import zipfile, os

zip_path = "/content/2024 IPL DATA.zip"
extract_path = "ipl_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

files = sorted(os.listdir(extract_path))
print("Matches:", len(files))

Matches: 1


In [29]:
import json
import numpy as np
from collections import defaultdict

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [34]:
player_stats = defaultdict(lambda: {
    "runs": 0,
    "balls": 0,
    "outs": 0,
    "matches": 0,
    "wickets": 0,
    "runs_conceded": 0,
    "balls_bowled": 0
})

def update_player_stats(data):
    for inn_data in data["innings"]:
        # Ensure inn_data is a dictionary and contains 'overs'
        if not isinstance(inn_data, dict) or "overs" not in inn_data:
            continue

        for over_data in inn_data["overs"]:
            # Ensure over_data is a dictionary and contains 'deliveries'
            if not isinstance(over_data, dict) or "deliveries" not in over_data:
                continue

            for ball in over_data["deliveries"]:
                # Ensure ball is a dictionary
                if not isinstance(ball, dict):
                    continue

                batter = ball.get("batter")
                bowler = ball.get("bowler")
                runs_data = ball.get("runs", {})
                runs = runs_data.get("batter", 0)
                total_runs_conceded = runs_data.get("total", 0)

                # batting stats
                if batter:
                    player_stats[batter]["runs"] += runs
                    player_stats[batter]["balls"] += 1

                # wicket
                if "wickets" in ball:
                    for w in ball["wickets"]:
                        # Ensure bowler and batter exist before updating stats
                        if bowler: player_stats[bowler]["wickets"] += 1
                        if batter: player_stats[batter]["outs"] += 1

                # bowling stats
                if bowler:
                    player_stats[bowler]["runs_conceded"] += total_runs_conceded
                    player_stats[bowler]["balls_bowled"] += 1

In [31]:
def get_team_players(data, team):
    return data["info"]["players"][team]

def compute_team_features(players):
    bat_avg, sr_list = [], []
    bowl_wkts, eco_list = [], []

    for p in players:
        stats = player_stats[p]

        # batting
        if stats["balls"] > 0:
            avg = stats["runs"] / max(1, stats["outs"])
            sr = (stats["runs"] / stats["balls"]) * 100

            bat_avg.append(avg)
            sr_list.append(sr)

        # bowling
        if stats["balls_bowled"] > 0:
            eco = (stats["runs_conceded"] / stats["balls_bowled"]) * 6
            wkts = stats["wickets"]

            eco_list.append(eco)
            bowl_wkts.append(wkts)

    # aggregate safely
    def safe(x): return np.mean(x) if len(x) > 0 else 0

    return [
        safe(bat_avg),
        safe(sr_list),
        safe(bowl_wkts),
        safe(eco_list),
        len(players)  # team depth
    ]

In [36]:
X, y = [], []

# Correctly iterate over the files inside the '2024 IPL DATA' directory
data_files_directory = os.path.join(extract_path, '2024 IPL DATA')
for file_name in sorted(os.listdir(data_files_directory)):
    path = os.path.join(data_files_directory, file_name)

    # Skip if it's not a file (e.g., if there are other subdirectories)
    if not os.path.isfile(path):
        continue

    with open(path) as f:
        data = json.load(f)

    info = data["info"]

    if "outcome" not in info or "winner" not in info["outcome"]:
        continue

    t1, t2 = info["teams"]
    winner = info["outcome"]["winner"]

    players1 = get_team_players(data, t1)
    players2 = get_team_players(data, t2)

    # Removed the 'skip if no prior data' logic for now to ensure dataset population
    # With only one file, this condition would always be met, resulting in an empty dataset.
    # The 'compute_team_features' function is robust enough to handle empty initial stats.

    f1 = compute_team_features(players1)
    f2 = compute_team_features(players2)

    features = f1 + f2
    label = 1 if winner == t1 else 0

    X.append(features)
    y.append(label)

    # Update player_stats with this match's data for potential future matches
    update_player_stats(data)

X = np.array(X)
y = np.array(y)

print("Dataset:", X.shape)

Dataset: (71, 10)


In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (56, 10)
X_test shape: (15, 10)
y_train shape: (56,)
y_test shape: (15,)


In [38]:

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=6, min_samples_leaf=2, n_estimators=200,
                       random_state=42)

In [39]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.6666666666666666
ROC-AUC: 0.6785714285714286

Classification Report:

              precision    recall  f1-score   support

           0       0.60      0.86      0.71         7
           1       0.80      0.50      0.62         8

    accuracy                           0.67        15
   macro avg       0.70      0.68      0.66        15
weighted avg       0.71      0.67      0.66        15

